# Materialized CE v3 — timeout-visible routing

**Prepared research candidate; new v3 has not run on a GPU. No performance, quality, 5% useful-kernel-share, score, or admission is established.**

This notebook contains the exact reviewed `candidate.zip` and source hashes. It reuses an **existing** official checkout at commit `c7c2fcd15972ce8b898d8d3ff732ee192c817ceb`, prepared public data/model cache, and exact installed packages. It never installs, downloads, submits, deploys, or edits the original checkout. Only its fresh staging copy's `solution/` is replaced.

V3 retains the five original numerical kernel bodies and native SUM / supplied-denominator / GradScaler order. It removes one duplicated denominator host read, uses the captured native loss for internal fallback, and flushes compact route evidence before the first JIT and at loss/forward/backward boundaries. A backward enqueue is **not** proof of completed CUDA work. Official CUDA-event timing after two warmup steps remains the only share evidence.

## Run later in an authorized prepared cloud runtime

Edit the paths below, choose `GCP` or `KAGGLE`, and set `execute_gpu=True` only when actual GPU testing is authorized. A nonblocking canonical `/tmp/soup-gpu0.lock` covers the entire run; a busy lane fails immediately. The shared `/tmp/soup-gpu0.lock.blocked` quarantine is checked before and after lock acquisition; an owner-tagged guard is durably created before the first GPU operation and survives abrupt runner death. Only the same owner clears it after normal verified success; any failure/timeout/interruption retains it for explicit operator reconciliation. GPU inventory runs only in a short-lived subprocess, which exits before the idle check. Run All also fails closed on missing assets, version drift, non-idle GPUs, or anything other than one visible Tesla T4. GCP always measures its own baseline; it never imports Kaggle's historical 79-second cutoff.

The public model is fixed at revision `989aa7980e4cf806f80c7fef2b1adb7bc71aa306`; `refs/main` drift is rejected. An actual-class v3 integration gate precedes the unchanged official repeat2 harness in base/solution/solution/base order. Each measured native process has fresh state/output/compiler caches. The official harness records complete command wall time, external 100ms nvidia-smi peak, public quality, all Soup outputs, and own-kernel timing. Failed/partial/timeout runs have no normalized comparison or score. All official checks run only after a verified complete pair. Final Kaggle admission remains separate.


In [ ]:
# Paths are operator configuration, not assumptions about an existing session.
# No GPU work occurs unless this explicit execution flag is true.
CONFIG = {'execute_gpu': False, 'backend': 'GCP', 'official_repo': '/prepared/soup-kernel-challenge', 'data_dir': '/prepared/public-data', 'hf_hub_cache': '/prepared/hf_hub', 'work_root': '/output/ce-v3-research'}
print("Prepared candidate; execute_gpu =", CONFIG["execute_gpu"])


## Exact embedded artifact and runner
The ZIP is checked against its SHA256 and every source-file hash before staging. No notebook-dependent optimization behavior is used: the worker receives exactly this `solution/` bundle.

In [ ]:
import base64, hashlib, json
BUNDLE_B64 = '...'
BUNDLE_SHA256 = 'b2e2622bb95d7d01bd69ae525d2d2098f42f3e4d43c807f058635ec60a20ed34'
SOURCE_MANIFEST = {'solution/__init__.py': '6e79458adeb6f5eea14c48f4d0fe4f138a09d377fb7206ff26a0e96ae0e468d6', 'solution/adapter.py': '2cf4c09d6450dcdaf829da2921e379475aaf10c499f284f6c17b52f6da4c0ac0', 'solution/materialized_ce.py': '79628d5ca19c8999227832398293f2530db20e191674b22323c88b446021f705', 'solution/numerical_gates.py': 'e23c0c7c87967392f95432d2b3f3062ae7798aa262efb58dde88c8c254bed990', 'solution/telemetry.py': 'f98a5212f85368fdad47efa2f4a800511269bae800460f09847318bb799d0fb2', 'solution/test_cpu_contract.py': '2b177aeca4b6e0afc6cb5fcb264f53221391edbed67fada7b5252ee179b5e18f', 'solution/verify_integration.py': 'bcc16afce2587811b8f4614b07128bff89a84b143c6e17b1d417f44437e93129'}
OFFICIAL_MANIFEST = {'harness/_train.py': '45a62642dceab32cf3efcda504de87a9c25ea28b119b1806f56d14ff1146a89d', 'harness/check_rules.py': '879ab1861674f6e714eadd00aaf044235d98526dd75fab1035deec3401121ef5', 'harness/common.py': 'c83066f24f266dab62ebcc9c2d44f5a65902760b53da912e65ebfe511579c2db', 'harness/eval_adapter.py': '47cddbb8e4579c69f9535b099ff2371b1c3af08bfbd93d14bf79aadd74d38482', 'harness/example_reference.py': '6a11055f4a1822e53971c228c6e3e3aa6b88edd9eb6db81a388e26a70a030ceb', 'harness/run_task.py': 'd935a8959f348977313f2eef36870ff2699b9ba013a7978918ff35e148968a79', 'harness/score.py': '731c9b644dcc4069b0b069616f438064d1f7f376a202feb15201fd910ec688c5', 'task/prepare_data.py': 'a8543f11a55aff80dbdc714a0e62391244a5296616eea2e34dc2eb3ff3ec18f7', 'task/task.yaml': 'dd70c739fcd143e06aef3bb1efcc9386f682bf1de26f30e08409d86a97ff47ce', 'tests/conftest.py': 'a82eabab468ea0ef1e003cccd7c3827ce2f4d00d09c32948a34fb48221f7337a', 'tests/test_callback.py': '1919303a3670c0a33abbfce9b33eab0913bb64a3a21c3312d86531c32b5bf490', 'tests/test_config.py': 'db808c44470d144b7cf18b7329492f41f76e37cfc24306e7cba0488bca9d787c', 'tests/test_formats.py': '5d656ed50c59f794e934473d459ea85635f892ba028e192414474e9b5d65919e', 'tests/test_loader.py': '3dad1bea2d33aab97e52b7cd44888d768bc9519af44fbe6cf3e96c260026632f', 'tests/test_trainer_init.py': 'b7608d62a40983bf7c9e5b270e0e527b641a837e462d6df2909fec88f607e821'}
RUNNER_SOURCE = '"""Self-contained prepared-runtime runner. No download/install/deployment/submission.\n\nThis file is embedded byte-for-byte in candidate.ipynb. Executing run() is a\nfuture GPU operation and is not part of package preparation or CPU validation.\n"""\nfrom __future__ import annotations\nimport argparse\nfrom contextlib import contextmanager\nimport hashlib\nimport importlib.metadata\nimport io\nimport json\nimport math\nimport os\nfrom pathlib import Path\nimport shutil\nimport subprocess\nimport sys\nimport tarfile\nimport time\nimport uuid\nimport zipfile\n\nCOMMIT = "c7c2fcd15972ce8b898d8d3ff732ee192c817ceb"\nCANDIDATE_ID = "soup-ce-v3-routing-20261030"\nMODEL_REVISION = "989aa7980e4cf806f80c7fef2b1adb7bc71aa306"\nPINS = {"torch":"2.11.0+cu128","triton":"3.6.0","soup-cli":"0.77.0",\n        "transformers":"5.16.1","peft":"0.20.0","trl":"0.29.1",\n        "bitsandbytes":"0.50.2","accelerate":"1.14.0","datasets":"4.8.5"}\nDATA_SHA256 = {"train.jsonl":"93d036d912bd81c337dc29893e13a76f36161d96f896ef4e02492e2f1a21a0dd",\n               "eval_public.jsonl":"b7f85b78ba3186dce4deab7cfb8bf8dcf7998e1517c5749ac0f5005886c6f70e"}\n\n\ndef sha(path):\n    h = hashlib.sha256()\n    with Path(path).open("rb") as source:\n        while chunk := source.read(1024 * 1024): h.update(chunk)\n    return h.hexdigest()\n\n\ndef save(path, value):\n    Path(path).write_text(json.dumps(value, indent=2, allow_nan=False) + "\\n")\n\n\ndef shell(args, *, cwd=None, env=None):\n    return subprocess.check_output(args, cwd=cwd, env=env, text=True).strip()\n\n\ndef _proc_identity(pid):\n    try:\n        # comm can contain spaces: parse from the final \')\' rather than split it.\n        fields=Path(f"/proc/{pid}/stat").read_text().rsplit(")",1)[1].split()\n        return int(fields[1]),fields[19]  # parent PID, process start-time ticks\n    except (OSError,ValueError,IndexError): return None\n\n\ndef _process_table():\n    table={}\n    for path in Path("/proc").iterdir():\n        if not path.name.isdigit(): continue\n        try:\n            fields=(path/"stat").read_text().rsplit(")",1)[1].split()\n            table[int(path.name)]={"ppid":int(fields[1]),"start":fields[19],"state":fields[0]}\n        except (OSError,ValueError,IndexError): pass\n    return table\n\n\ndef terminate_owned_tree(proc, root_start, tracked=None):\n    """Freeze first, then discover/freeze the tree to prevent the census/spawn race.\n\n    Start-time identity guards PID reuse. This runs inside the notebook, not a\n    global subreaper: it never adopts or kills unrelated notebook children.\n    Quarantine always remains after interruption, even if this cleanup verifies.\n    """\n    import signal\n    owned=dict(tracked or {});owned[proc.pid]=root_start;stopped=set();errors=[]\n    def signal_owned(pid,start,sig):\n        identity=_proc_identity(pid)\n        if identity is None or identity[1]!=start: return\n        try: os.kill(pid,sig)\n        except ProcessLookupError: pass\n        except OSError as error: errors.append(f"{pid}:{type(error).__name__}")\n    # The root cannot create a child after this stop is delivered. Iteratively\n    # stop existing descendants before taking the final complete census.\n    signal_owned(proc.pid,root_start,signal.SIGSTOP)\n    deadline=time.monotonic()+5\n    stable=0\n    while stable<2 and time.monotonic()<deadline:\n        table=_process_table();added=False\n        live_owned={pid for pid,start in owned.items() if pid in table and table[pid]["start"]==start}\n        for pid,record in table.items():\n            if record["ppid"] in live_owned and pid not in owned:\n                owned[pid]=record["start"];added=True\n        for pid,start in list(owned.items()):\n            if pid not in stopped:\n                signal_owned(pid,start,signal.SIGSTOP);stopped.add(pid)\n        # Stopped roots preserve ancestry until every live branch is stopped.\n        table=_process_table()\n        unsettled=[pid for pid,start in owned.items() if pid in table\n                   and table[pid]["start"]==start and table[pid]["state"] not in ("T","t","Z","X")]\n        stable=stable+1 if not added and not unsettled else 0\n        time.sleep(0.01)\n    for pid,start in list(owned.items()):\n        if pid!=proc.pid: signal_owned(pid,start,signal.SIGKILL)\n    signal_owned(proc.pid,root_start,signal.SIGKILL)\n    try: proc.wait(timeout=10)\n    except subprocess.TimeoutExpired: errors.append("root_wait_timeout")\n    deadline=time.monotonic()+5\n    while True:\n        table=_process_table()\n        remaining=[pid for pid,start in owned.items() if pid in table\n                   and table[pid]["start"]==start and table[pid]["state"] not in ("Z","X")]\n        if not remaining or time.monotonic()>=deadline: break\n        time.sleep(0.02)\n    report={"known_owned_count":len(owned),"known_owned_pids":sorted(owned),\n            "remaining_live_known_pids":remaining,"errors":errors,\n            "frozen_census_stable":stable>=2,\n            "verified_known_tree_terminated":not remaining and not errors and stable>=2,\n            "quarantine_required":True,\n            "limit":"No SIGKILL/power-loss or already-unobserved orphan guarantee; operator reconciliation remains required."}\n    return report\n\n\ndef mark_quarantine(path, reason):\n    """Immutable first incident; never auto-clear or overwrite another incident."""\n    path=Path(path)\n    payload={"schema":1,"status":"OPERATOR_RECONCILIATION_REQUIRED","reason":reason,\n             "runner_pid":os.getpid(),"utc_epoch_s":time.time(),"automatic_clear":False}\n    try:\n        fd=os.open(path,os.O_CREAT|os.O_EXCL|os.O_WRONLY|getattr(os,"O_NOFOLLOW",0),0o600)\n    except FileExistsError: return\n    with os.fdopen(fd,"w") as file:\n        file.write(json.dumps(payload,sort_keys=True)+"\\n");file.flush();os.fsync(file.fileno())\n\n\ndef create_active_marker(path, owner_token):\n    """Durable owner-tagged fail-closed guard, installed before the first GPU call."""\n    payload={"schema":1,"status":"ACTIVE_OWNER_GUARD","owner_token":owner_token,\n             "runner_pid":os.getpid(),"utc_epoch_s":time.time(),\n             "recovery":"If not removed by normal verified owner completion, operator reconciliation is required."}\n    fd=os.open(path,os.O_CREAT|os.O_EXCL|os.O_WRONLY|getattr(os,"O_NOFOLLOW",0),0o600)\n    with os.fdopen(fd,"w") as file:\n        file.write(json.dumps(payload,sort_keys=True)+"\\n");file.flush();os.fsync(file.fileno())\n    directory=os.open(str(Path(path).parent),os.O_RDONLY|getattr(os,"O_DIRECTORY",0))\n    try: os.fsync(directory)\n    finally: os.close(directory)\n\n\ndef clear_owned_marker(path, owner_token):\n    """Only the same process/token may remove its marker on verified success."""\n    path=Path(path)\n    try:\n        fd=os.open(path,os.O_RDONLY|getattr(os,"O_NOFOLLOW",0))\n        with os.fdopen(fd,"r") as file:\n            identity=os.fstat(file.fileno())\n            value=json.loads(file.read(4096))\n        current=path.lstat()\n    except (OSError,ValueError,TypeError): return False\n    if value.get("owner_token")!=owner_token or value.get("runner_pid")!=os.getpid(): return False\n    if (identity.st_dev,identity.st_ino)!=(current.st_dev,current.st_ino): return False\n    path.unlink()\n    directory=os.open(str(path.parent),os.O_RDONLY|getattr(os,"O_DIRECTORY",0))\n    try: os.fsync(directory)\n    finally: os.close(directory)\n    return True\n\n\ndef observe_owned(owned):\n    table=_process_table();changed=True\n    while changed:\n        changed=False\n        live={pid for pid,start in owned.items() if pid in table and table[pid]["start"]==start}\n        for pid,value in table.items():\n            if value["ppid"] in live and pid not in owned:\n                owned[pid]=value["start"];changed=True\n    return [pid for pid,start in owned.items() if pid in table and table[pid]["start"]==start\n            and table[pid]["state"] not in ("Z","X")]\n\n\n@contextmanager\ndef interruption_handlers():\n    import signal\n    def interrupted(signum, frame):\n        raise KeyboardInterrupt(f"runner interrupted by signal {signum}")\n    old={sig:signal.getsignal(sig) for sig in (signal.SIGINT,signal.SIGTERM)}\n    try:\n        for sig in old: signal.signal(sig,interrupted)\n        yield\n    finally:\n        for sig,handler in old.items(): signal.signal(sig,handler)\n\n\ndef execute(command, *, cwd, env, logfile, timeout,\n            quarantine_path="/tmp/soup-gpu0.lock.blocked"):\n    started=time.monotonic()\n    with interruption_handlers(), Path(logfile).open("w") as log:\n        import signal\n        # Defer parent interrupts until proc+identity are registered. A tiny exec\n        # launcher unblocks them in the child; no unsafe threaded preexec_fn.\n        original_mask=signal.pthread_sigmask(signal.SIG_BLOCK,{signal.SIGINT,signal.SIGTERM})\n        proc=None;root_start=None;tracked={}\n        launcher="import os,signal,sys; signal.pthread_sigmask(signal.SIG_UNBLOCK,{signal.SIGINT,signal.SIGTERM}); os.execvpe(sys.argv[1],sys.argv[1:],os.environ)"\n        try:\n            proc=subprocess.Popen([sys.executable,"-c",launcher,*command],cwd=cwd,env=env,stdout=log,\n                                  stderr=subprocess.STDOUT,start_new_session=True)\n            identity=_proc_identity(proc.pid)\n            root_start=identity[1] if identity else None\n            tracked[proc.pid]=root_start\n            signal.pthread_sigmask(signal.SIG_SETMASK,original_mask)\n            deadline=time.monotonic()+timeout\n            while True:\n                live=observe_owned(tracked)\n                code=proc.poll()\n                if code is not None:\n                    survivors=[pid for pid in observe_owned(tracked) if pid!=proc.pid]\n                    if survivors: raise RuntimeError("Owned descendants survived normal command exit")\n                    save(str(logfile)+".processes.json",{"normal_exit":True,"exit_code":code,\n                         "known_owned_count":len(tracked),"remaining_live_known_pids":[]})\n                    break\n                if time.monotonic()>=deadline: raise subprocess.TimeoutExpired(command,timeout)\n                time.sleep(min(0.1,max(0.001,deadline-time.monotonic())))\n        except BaseException as error:\n            import signal\n            reason="outer_timeout" if isinstance(error,subprocess.TimeoutExpired) else type(error).__name__\n            if proc is None:\n                signal.pthread_sigmask(signal.SIG_SETMASK,original_mask)\n                raise\n            # A durable shared marker precedes cleanup and remains even when\n            # cleanup succeeds. No other notebook may begin pending reconciliation.\n            mask=signal.pthread_sigmask(signal.SIG_BLOCK,{signal.SIGINT,signal.SIGTERM})\n            try:\n                marker_error=None\n                try: mark_quarantine(quarantine_path,reason)\n                except OSError as failure: marker_error=f"{type(failure).__name__}: {failure}"\n                cleanup=terminate_owned_tree(proc,root_start,tracked)\n                cleanup["quarantine_write_error"]=marker_error\n                save(str(logfile)+".cleanup.json",cleanup)\n                log.write("\\nOUTER_INTERRUPTION "+json.dumps({"reason":reason,"cleanup":cleanup})+"\\n");log.flush()\n                if not cleanup["verified_known_tree_terminated"] or marker_error:\n                    raise RuntimeError("Cleanup or quarantine persistence could not be confirmed; operator reconciliation required") from error\n                if isinstance(error,subprocess.TimeoutExpired): code=-9\n                else: raise\n            finally: signal.pthread_sigmask(signal.SIG_SETMASK,original_mask)\n        finally:\n            signal.pthread_sigmask(signal.SIG_SETMASK,original_mask)\n    return code,time.monotonic()-started\n\n\ndef validate_zip(raw, manifest):\n    with zipfile.ZipFile(io.BytesIO(raw)) as archive:\n        names = archive.namelist()\n        if len(names) != len(set(names)) or set(names) != set(manifest):\n            raise ValueError("ZIP member set differs from reviewed source manifest")\n        for name in names:\n            p = Path(name)\n            if not name.startswith("solution/") or p.is_absolute() or ".." in p.parts:\n                raise ValueError("Unsafe ZIP member")\n            if hashlib.sha256(archive.read(name)).hexdigest() != manifest[name]:\n                raise ValueError("ZIP member hash mismatch: " + name)\n    return True\n\n\ndef parse_routes(log):\n    records=[]; partial=0\n    if not Path(log).exists(): return {"status":"UNKNOWN", "records":[], "partial_lines":0}\n    for line in Path(log).read_text(errors="replace").splitlines():\n        prefix="SOUP_CE_V3_EVENT "\n        if prefix not in line: continue\n        try: record=json.JSONDecoder().raw_decode(line.split(prefix,1)[1])[0]\n        except (ValueError, TypeError): partial+=1; continue\n        if isinstance(record,dict): records.append(record)\n    return {"status":"OBSERVED_ROUTE_RECORDS" if records else "UNKNOWN",\n            "records":records, "partial_lines":partial,\n            "last_record":records[-1] if records else None,\n            "interpretation":"Route and enqueue metadata only. Not GPU timing/share or completion."}\n\n\ndef public_model_manifest(cache):\n    """Same pinned cache identity/boundary as the prepared queue asset contract."""\n    root=Path(cache)/"models--Qwen--Qwen2.5-1.5B-Instruct"\n    for directory in (root,root/"refs",root/"snapshots",root/"blobs"):\n        if directory.is_symlink() or not directory.is_dir():\n            raise RuntimeError("Selected public cache directories must exist and be real")\n    ref=root/"refs/main"\n    if ref.is_symlink() or not ref.is_file() or ref.read_text().strip()!=MODEL_REVISION:\n        raise RuntimeError("Prepared refs/main must identify the exact pinned model revision")\n    snapshot=root/"snapshots"/MODEL_REVISION\n    if snapshot.is_symlink() or not snapshot.is_dir():\n        raise RuntimeError("Pinned snapshot missing; no download or link reconstruction attempted")\n    files={str(ref.relative_to(cache)):sha(ref)};memo={}\n    for path in sorted(snapshot.rglob("*")):\n        if path.is_dir():\n            if path.is_symlink(): raise RuntimeError("Linked snapshot directory rejected")\n            continue\n        resolved=path.resolve(strict=True)\n        if not resolved.is_relative_to((root/"blobs").resolve()) or not resolved.is_file():\n            raise RuntimeError("Snapshot file must resolve only inside selected public model blobs")\n        if resolved not in memo: memo[resolved]=sha(resolved)\n        files[str(path.relative_to(cache))]=memo[resolved]\n    if len(files)<2 or not (snapshot/"config.json").is_file(): raise RuntimeError("Prepared model snapshot incomplete")\n    return {"model_id":"Qwen/Qwen2.5-1.5B-Instruct","revision":MODEL_REVISION,"files":files}\n\n\ndef completed_result(result):\n    """Never infer completion from wrapper exit, last progress, or nested step 64 alone."""\n    reasons=[]\n    runs=result.get("runs",{})\n    task=result.get("task",{})\n    if task.get("repeats")!=2 or task.get("seed")!=1 or task.get("only") is not None:\n        reasons.append("Expected fixed repeat2/seed1 paired task")\n    for side in ("base","solution"):\n        entries=runs.get(side,[])\n        if len(entries)!=2: reasons.append(side+": expected exactly two native runs")\n        for entry in entries:\n            metrics=entry.get("metrics",{})\n            if entry.get("exit_code") != 0 or metrics.get("exit_code") != 0:\n                reasons.append(side+": native process incomplete/failed")\n            if entry.get("global_step") != 64 or metrics.get("steps") != 64:\n                reasons.append(side+": not all fixed 64 steps verified")\n            if not entry.get("adapter_config") or not entry.get("adapter_tensors"):\n                reasons.append(side+": adapter artifact missing")\n            for key in ("wall_s","gpu_peak_mib"):\n                val=entry.get(key)\n                if isinstance(val,bool) or not isinstance(val,(int,float)) or not math.isfinite(val) or val <= 0:\n                    reasons.append(side+": missing complete "+key)\n        evaluation=result.get("evals",{}).get(side,{})\n        loss=evaluation.get("mean_loss")\n        if evaluation.get("exit_code") != 0 or isinstance(loss,bool) or not isinstance(loss,(int,float)) or not math.isfinite(loss):\n            reasons.append(side+": public quality missing/failed")\n    if result.get("packages_changed") != []: reasons.append("packages changed or not inventoried")\n    return not reasons,reasons\n\n\ndef _run_locked(config, bundle_bytes, source_manifest, official_manifest):\n    if config.get("execute_gpu") is not True:\n        raise RuntimeError("Preparation only. Set execute_gpu=True only for an authorized cloud T4 run.")\n    if config.get("backend") not in ("GCP","KAGGLE"):\n        raise ValueError("backend must explicitly be GCP or KAGGLE; GCP is never official Kaggle admission")\n    repo=Path(config["official_repo"]).resolve(); data=Path(config["data_dir"]).resolve()\n    cache=Path(config["hf_hub_cache"]).resolve(); work=Path(config["work_root"]).resolve()\n    for path in (repo,data,cache):\n        if not path.is_dir(): raise FileNotFoundError(f"Prepared asset missing: {path}; no download/bootstrap attempted")\n    # Existing packages only. Never mutate them or silently select a different wheel.\n    versions={name:importlib.metadata.version(name) for name in PINS}\n    if sys.version_info[:3] != (3,13,15) or versions != PINS:\n        raise RuntimeError("Exact prepared Python3.13.15/direct package pins required: "+repr(versions))\n    importlib.metadata.version("pytest")  # required by unchanged official regression runner\n    if shell(["git","-C",str(repo),"rev-parse","HEAD"]) != COMMIT:\n        raise RuntimeError("Official starter commit mismatch")\n    status=shell(["git","-C",str(repo),"status","--porcelain","--untracked-files=all","--","harness","task","tests"])\n    if status: raise RuntimeError("Official harness/task/tests must be unchanged: "+status)\n    for relative, digest in official_manifest.items():\n        if sha(repo/relative) != digest: raise RuntimeError("Official source hash mismatch: "+relative)\n    for name,digest in DATA_SHA256.items():\n        if sha(data/name) != digest: raise RuntimeError("Prepared public data hash mismatch: "+name)\n    validate_zip(bundle_bytes,source_manifest)\n    # No torch imported into this notebook process; GPU visibility is fixed before\n    # all actual-class, preflight and measured subprocess imports.\n    env=dict(os.environ,CUDA_VISIBLE_DEVICES="0",CUDA_DEVICE_ORDER="PCI_BUS_ID",\n             PYTORCH_CUDA_ALLOC_CONF="expandable_segments:True",PYTHONUNBUFFERED="1",\n             TOKENIZERS_PARALLELISM="false",SOUP_NO_AUDIT_LOG="1",HF_HUB_OFFLINE="1",\n             HF_DATASETS_OFFLINE="1",TRANSFORMERS_OFFLINE="1",HF_HUB_CACHE=str(cache),\n             HF_HOME=str(cache.parent))\n    runroot=work/(CANDIDATE_ID+"-"+uuid.uuid4().hex)\n    runroot.mkdir(parents=True,exist_ok=False)\n    report={"candidate_id":CANDIDATE_ID,"status":"PREPARING","backend":config["backend"],\n            "official_commit":COMMIT,"measured_baseline_id":None,"score":None,\n            "automatic_admission":"UNKNOWN","runroot":str(runroot),\n            "source_sha256":hashlib.sha256(bundle_bytes).hexdigest(),\n            "environment":{"python":sys.version,"packages":versions},\n            "limit":"Research pair only; official Kaggle admission and jury review remain separate."}\n    save(runroot/"receipt.json",report)\n    try:\n        gpu_code="""import json,torch\nassert torch.version.cuda==\'12.8\'\nassert torch.cuda.device_count()==1\nassert torch.cuda.get_device_name(0)==\'Tesla T4\'\nassert torch.cuda.get_device_capability(0)==(7,5)\nprint(json.dumps({\'visible_count\':torch.cuda.device_count(),\'device\':torch.cuda.get_device_name(0),\'cuda\':torch.version.cuda}))\n"""\n        report["gpu"]=json.loads(shell([sys.executable,"-c",gpu_code],env=env))\n        inventory=shell(["nvidia-smi","--query-gpu=index,name,driver_version,memory.used,memory.total","--format=csv,noheader,nounits"],env=env)\n        report["nvidia_smi_inventory"]=inventory\n        active=shell(["nvidia-smi","--query-compute-apps=pid","--format=csv,noheader,nounits"],env=env)\n        if any(line.strip().isdigit() for line in active.splitlines()):\n            raise RuntimeError("A compute process is still using a physical GPU, possibly this notebook; no measurement starts")\n        total_used=sum(int(line.split(",")[-2].strip()) for line in inventory.splitlines())\n        if total_used >= 1000: raise RuntimeError("GPU(s) are not idle; never stop unrelated workloads")\n        report["cpu_affinity"]=sorted(os.sched_getaffinity(0)) if hasattr(os,"sched_getaffinity") else None\n        # The queue-prepared public model revision and refs/main are fixed;\n        # actual byte hashing is additional binding, not permission to use drift.\n        cache_manifest=public_model_manifest(cache)\n        cache_code="from huggingface_hub import snapshot_download; print(snapshot_download(\'Qwen/Qwen2.5-1.5B-Instruct\', local_files_only=True, token=False))"\n        resolved_snapshot=Path(shell([sys.executable,"-c",cache_code],env=env)).resolve()\n        expected_snapshot=(cache/"models--Qwen--Qwen2.5-1.5B-Instruct"/"snapshots"/MODEL_REVISION).resolve()\n        if resolved_snapshot!=expected_snapshot: raise RuntimeError("Offline main resolved a different model snapshot")\n        save(runroot/"model_cache_manifest.json",cache_manifest)\n        report["model_cache_manifest_sha256"]=sha(runroot/"model_cache_manifest.json")\n        report["data_sha256"]=DATA_SHA256\n        # Fresh clean source tree; original checkout, prior solutions and outputs\n        # are preserved. Only solution/ is replaced in this private staging copy.\n        stage=runroot/"source"; stage.mkdir()\n        raw=subprocess.check_output(["git","-C",str(repo),"archive",COMMIT])\n        with tarfile.open(fileobj=io.BytesIO(raw)) as archive:\n            for member in archive.getmembers():\n                p=Path(member.name)\n                if p.is_absolute() or ".." in p.parts or not (member.isfile() or member.isdir()):\n                    raise ValueError("Unsafe official archive entry")\n            archive.extractall(stage,filter="data")\n        shutil.rmtree(stage/"solution")\n        with zipfile.ZipFile(io.BytesIO(bundle_bytes)) as archive: archive.extractall(stage)\n        for name,digest in source_manifest.items():\n            if sha(stage/name) != digest: raise RuntimeError("Staged candidate differs: "+name)\n        for name,digest in official_manifest.items():\n            if sha(stage/name) != digest: raise RuntimeError("Staged official file differs: "+name)\n        (stage/"data").mkdir(exist_ok=True)\n        for name in DATA_SHA256: shutil.copyfile(data/name,stage/"data"/name)\n        save(runroot/"source_manifest.json",source_manifest)\n        save(runroot/"official_manifest.json",official_manifest)\n        (runroot/"candidate.zip").write_bytes(bundle_bytes)\n        # Fresh process/cache for a new v3 actual-class gate; previous v2 results\n        # are historical and cannot satisfy this gate. No training modifications.\n        preflight_env=dict(env,TRITON_CACHE_DIR=str(runroot/"preflight_triton_cache"),\n                           TORCHINDUCTOR_CACHE_DIR=str(runroot/"preflight_inductor_cache"))\n        preflight_cmd=[sys.executable,"-m","solution.verify_integration","--out",str(runroot/"integration_preflight.json")]\n        preflight_code,preflight_wall=execute(preflight_cmd,cwd=stage,env=preflight_env,\n            logfile=runroot/"integration_preflight.log",timeout=300)\n        report["preflight_wall_s"]=preflight_wall\n        report["preflight_exit_code"]=preflight_code\n        if preflight_code != 0: raise RuntimeError("v3 actual-class gate failed or timed out; no training follows")\n        preflight=json.loads((runroot/"integration_preflight.json").read_text())\n        if preflight.get("status") != "passed_gpu_integration_preflight": raise RuntimeError("v3 gate status missing")\n        # Unchanged official harness: own same-session baseline, candidate, full\n        # subprocess wall, external nvidia-smi, fresh state/cache, public eval,\n        # and CUDA-event own kernel timing after exactly two warmup steps.\n        # 900s is the official default, not the censored Kaggle79s screen budget.\n        report["native_timeout_s"]=900\n        report["measured_order"]=["base","solution","solution","base"]\n        cmd=[sys.executable,"harness/run_task.py","--repeats","2","--seed","1",\n             "--timeout","900","--out",str(runroot/"runs")]\n        harness_code,wrapper_wall=execute(cmd,cwd=stage,env=env,\n            logfile=runroot/"harness.log",timeout=9000)\n        report["harness_exit_code"]=harness_code\n        report["complete_wrapper_wall_s"]=wrapper_wall\n        report["outer_watchdog_s"]=9000\n        result_path=runroot/"runs"/"result.json"\n        if not result_path.exists(): raise RuntimeError("Official result absent; completed metrics unknown")\n        result=json.loads(result_path.read_text())\n        report["routes"]={p.parent.name:parse_routes(p) for p in (runroot/"runs").glob("solution_*/train.log")}\n        complete,reasons=completed_result(result)\n        report["native_pair_complete"]=complete\n        report["completion_blockers"]=reasons\n        report["raw_result_sha256"]=sha(result_path)\n        report["comparison"]=result.get("summary") if complete and harness_code == 0 else None\n        if not complete or harness_code != 0:\n            report["status"]="FAILED_OR_CENSORED_NO_SCORE"\n            return report\n        checks=[sys.executable,"harness/check_rules.py","--result",str(result_path)]\n        checks_code,checks_wall=execute(checks,cwd=stage,env=env,\n            logfile=runroot/"checks.log",timeout=900)\n        report["official_checks_exit_code"]=checks_code\n        report["official_checks_wall_s"]=checks_wall\n        report["official_checks_path"]=str(runroot/"runs"/"checks.json")\n        report["status"]="RESEARCH_PAIR_COMPLETE_REVIEW_REQUIRED"\n        # All ten raw checks may pass, but no automatic official admission or\n        # leaderboard score is manufactured from this public research run.\n        return report\n    except BaseException as error:\n        report.update(status="BLOCKED_OR_FAILED_NO_SCORE",error=f"{type(error).__name__}: {error}")\n        raise\n    finally:\n        # Original frozen files must remain unchanged even after failed runs.\n        if (runroot/"source").exists():\n            report["official_files_preserved"]=all((runroot/"source"/p).exists() and sha(runroot/"source"/p)==h for p,h in official_manifest.items())\n            report["candidate_source_preserved"]=all((runroot/"source"/p).exists() and sha(runroot/"source"/p)==h for p,h in source_manifest.items())\n            report["routes"]={p.parent.name:parse_routes(p) for p in (runroot/"runs").glob("solution_*/train.log")}\n        if report.get("official_files_preserved") is False or report.get("candidate_source_preserved") is False:\n            report.update(status="SOURCE_MUTATED_NO_SCORE",comparison=None)\n        save(runroot/"receipt.json",report)\n        print(json.dumps({"status":report["status"],"receipt":str(runroot/"receipt.json"),"score":None,"admission":"UNKNOWN"},indent=2))\n\n\ndef run(config, bundle_bytes, source_manifest, official_manifest):\n    """One canonical nonblocking GPU0 lock covers preparation, gates and pair."""\n    if config.get("execute_gpu") is not True:\n        raise RuntimeError("Preparation only. Set execute_gpu=True only for an authorized cloud T4 run.")\n    import fcntl\n    flags=os.O_CREAT | os.O_RDWR | getattr(os,"O_NOFOLLOW",0)\n    quarantine=Path("/tmp/soup-gpu0.lock.blocked")\n    if quarantine.exists() or quarantine.is_symlink():\n        raise RuntimeError("GPU0 is quarantined; operator reconciliation is required before any GPU work")\n    fd=os.open("/tmp/soup-gpu0.lock",flags,0o600)\n    try:\n        try: fcntl.flock(fd,fcntl.LOCK_EX | fcntl.LOCK_NB)\n        except BlockingIOError as error:\n            raise RuntimeError("GPU0 lane is owned by another process; no GPU work started") from error\n        if quarantine.exists() or quarantine.is_symlink():\n            raise RuntimeError("GPU0 quarantine appeared while acquiring lock; no GPU work started")\n        owner_token=uuid.uuid4().hex\n        create_active_marker(quarantine,owner_token)\n        with interruption_handlers():\n            try:\n                report=_run_locked(config,bundle_bytes,source_manifest,official_manifest)\n                # Failures retain the active guard. Only a clean complete pair,\n                # successful official checks, unchanged source and idle GPU can\n                # remove THIS process/token\'s marker. SIGKILL cannot remove it.\n                normal=(report.get("status")=="RESEARCH_PAIR_COMPLETE_REVIEW_REQUIRED"\n                        and report.get("official_checks_exit_code")==0\n                        and report.get("official_files_preserved") is True\n                        and report.get("candidate_source_preserved") is True)\n                if normal:\n                    active=shell(["nvidia-smi","--query-compute-apps=pid","--format=csv,noheader,nounits"])\n                    if any(line.strip().isdigit() for line in active.splitlines()):\n                        raise RuntimeError("GPU compute process survived normal finish; active owner guard retained")\n                    if not clear_owned_marker(quarantine,owner_token):\n                        raise RuntimeError("Owner marker changed or could not be cleared; no foreign marker removed")\n                    report["lane_guard"]="OWN_MARKER_CLEARED_AFTER_NORMAL_VERIFIED_SUCCESS"\n                else: report["lane_guard"]="RETAINED_OPERATOR_RECONCILIATION_REQUIRED"\n                if report.get("runroot"): save(Path(report["runroot"])/"receipt.json",report)\n                return report\n            except KeyboardInterrupt:\n                mark_quarantine(quarantine,"outer_runner_interruption")\n                raise\n    finally:\n        # The persistent blocked marker prevents reuse even after a lock release.\n        # Never unlink the lock inode; a failed/interrupted owner guard is not cleared here.\n        fcntl.flock(fd,fcntl.LOCK_UN)\n        os.close(fd)\n\n\nif __name__ == "__main__":\n    p=argparse.ArgumentParser(description=__doc__)\n    p.add_argument("--config",required=True)\n    args=p.parse_args()\n    root=Path(__file__).resolve().parent\n    run(json.loads(Path(args.config).read_text()),(root/"candidate.zip").read_bytes(),\n        json.loads((root/"source_manifest.json").read_text()),\n        json.loads((root/"official_manifest.json").read_text()))\n'
BUNDLE_BYTES = base64.b64decode(BUNDLE_B64, validate=True)
assert hashlib.sha256(BUNDLE_BYTES).hexdigest() == BUNDLE_SHA256
namespace = {"__name__": "prepared_ce_notebook_runner"}
exec(compile(RUNNER_SOURCE, "notebook_runner.py", "exec"), namespace)
namespace["validate_zip"](BUNDLE_BYTES, SOURCE_MANIFEST)
print("Bundle integrity verified:", BUNDLE_SHA256)


In [ ]:
# Future GPU execution: no package installation, asset download, queue submission,
# source/harness modification, hidden rows, or historical baseline substitution.
REPORT = namespace["run"](CONFIG, BUNDLE_BYTES, SOURCE_MANIFEST, OFFICIAL_MANIFEST)
print(json.dumps(REPORT, indent=2, allow_nan=False))


## Interpret evidence conservatively
- `receipt.json`: actual environment/source/data/cache bindings, native-pair completion, route records, and raw result/check paths. `score` is always null and automatic admission UNKNOWN.
- `runs/result.json`: unchanged official raw timing/memory/quality evidence. A timeout's wall/peak are censored, not a completed comparison.
- `runs/solution_1/train.log`: immediately flushed `SOUP_CE_V3_EVENT` records survive native process termination; an absent record is UNKNOWN. Core-native and adapter-native counters can describe the same fallback and must not be added.
- `runs/solution_1/metrics.json`: official own-kernel launch/event timing, when available. The useful-own-kernel requirement is >=5% after two warmup steps; Python counters never satisfy it.
- `runs/checks.json`: unchanged official check results. No leaderboard score is manufactured and a GCP pair cannot establish official Kaggle admission.

The old trial008 all-native routing and trial009 timeout remain failed historical variants. Previous v2 numerical preflight results are not relabeled as a v3 integration or whole-task pass. Instrumentation overhead is included in measurements and remains unknown until measured.
